In [ ]:
import os
import re
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from datetime import datetime
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from scipy.stats import linregress
from pathlib import Path
from abc import ABCMeta, abstractmethod
from time import time
import scipy.sparse as sp
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LinearRegression

In [ ]:
sys.path.append(os.path.abspath('..'))
from configs.config import *
from src.util import Logger, Util
from src.feature import *

In [ ]:
import importlib
import src.feature
importlib.reload(src.feature)
from src.feature import *

In [ ]:
pd.set_option("display.max_columns",500)
pd.set_option("display.max_rows", 500)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# 処理実行

In [ ]:
def run_blocks(feature_blocks):
    print('start run blocks...')
    with Timer(prefix='run test'):
        for block in feature_blocks:
            with Timer(prefix='\t- {}'.format(str(block))):
                feature = block.create_feature()

In [ ]:
feature_blocks = [
    Key(use_cache=False, save_cache=True, logger=None),
	Target(use_cache=False, save_cache=True, logger=None),
    CategoryFeature(use_cache=False, save_cache=True, logger=None),
	CareerFeature(use_cache=False, save_cache=True, logger=None),
	UdemyActivityFeature(use_cache=False, save_cache=True, logger=None),
    UdemyTimeseriesFeature(use_cache=True, save_cache=True, logger=None),
    UdemyTitleEmbedding(use_cache=False, save_cache=True, logger=None),
	UdemyIDEmbedding(use_cache=False, save_cache=True, logger=None),
	DxFeature(use_cache=False, save_cache=True, logger=None),
	HrFeature(use_cache=False, save_cache=True, logger=None),
	OvertimeWorkByMonthFeature(use_cache=False, save_cache=True, logger=None),
    OvertimeWorkByMonthTimeseriesFeature(use_cache=True, save_cache=True, logger=None),
	PositionHistoryFeature(use_cache=False, save_cache=True, logger=None),
]

In [ ]:
run_blocks(feature_blocks)

In [ ]:
list_ = [
    'Key',
    'Target',
    'CategoryFeature',
    'CareerFeature',
    'UdemyActivityFeature',
    'UdemyTimeseriesFeature',
    'UdemyTitleEmbedding',
    'UdemyIDEmbedding',
    'DxFeature',
    'HrFeature',
    'OvertimeWorkByMonthFeature',
    'OvertimeWorkByMonthTimeseriesFeature',
    'PositionHistoryFeature',
]
dict_shape = {}
for feature_name in list_:
    dict_shape[feature_name] = Util.load_feature(feature_name).shape
pd.DataFrame(dict_shape, index=['n_rows', 'n_cols']).T.sort_values('n_rows', ascending=False).sort_values('n_cols', ascending=False)

In [ ]:
df_udemy = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_udemy_activity.pkl"))

In [ ]:
df_udemy_target = df_udemy.copy()
df_udemy_target['開始年'] = df_udemy_target['開始日'].dt.to_period('Y')

# 社員番号、開始年ごとの受講数を集計して増加数を計算
df_udemy_target = df_udemy_target.groupby(['社員番号', '開始年']).size().reset_index(name='受講数')
# 社員番号、開始年のすべての組み合わせを作成
id = df_udemy_target['社員番号'].unique()
year = df_udemy_target['開始年'].unique()
df_udemy_target_all = pd.MultiIndex.from_product([id, year], names=['社員番号', '開始年']).to_frame(index=False)
df_udemy_target_all = df_udemy_target_all.merge(df_udemy_target, on=['社員番号', '開始年'], how='left')
# lagを計算
df_udemy_target_all.sort_values(['社員番号', '開始年'], inplace=True)
# lag特徴量を生成
lag=7
for i in range(1, lag + 1):
    df_udemy_target_all[f'ua_受講数_{i}_age'] = df_udemy_target_all.groupby('社員番号')['受講数'].shift(i)

# 最新行を抽出
df_udemy_lag = df_udemy_target_all.groupby('社員番号').tail(1).reset_index(drop=True)

# カラム整形
lag_cols = [f'ua_受講数_{i}_age' for i in range(1, lag + 1)]
df_udemy_lag = df_udemy_lag[['社員番号', '受講数'] + lag_cols]
df_udemy_lag = df_udemy_lag.rename(columns={'受講数': 'ua_受講数_0_age'})

# 前年との受講数の差分を計算
for i in range(0, lag):
    df_udemy_lag[f'ua_受講数_{i}_age_diff'] = df_udemy_lag[f'ua_受講数_{i}_age'] - df_udemy_lag[f'ua_受講数_{i+1}_age']

# プラスとマイナスの受講数の差分を計算
cols = [f'ua_受講数_{i}_age_diff' for i in range(0, lag)]
df_udemy_lag['ua_受講数_0_age_diff_plus'] = df_udemy_lag[cols].apply(lambda x: len(x[x > 0]), axis=1)
df_udemy_lag['ua_受講数_0_age_diff_minus'] = df_udemy_lag[cols].apply(lambda x: len(x[x < 0]), axis=1)

In [ ]:
df_udemy_lag

In [ ]:
df_udemy_target['開始年'].value_counts()

In [ ]:
def make_worker_hours_lag_features(df_overtime, lag=35):
    """
    社員別の過去労働時間（lag特徴量）を作成し、最新月の1行にまとめる。

    Parameters:
        df_overtime: DataFrame
            '社員番号', 'date', 'hours' を含むDataFrame
        lag: int
            生成する最大lag数（例：35であれば hours_1_age ～ hours_35_age）
    Returns:
        df_worker_lag: DataFrame
            社員番号ごとの最新行 + lag特徴量（hours_0_age ～ hours_{lag}_age）
    """
    df = df_overtime.copy()
    df = df.sort_values(['社員番号', 'date']).reset_index(drop=True)

    # lag特徴量を生成
    for i in range(1, lag + 1):
        df[f'hours_{i}_age'] = df.groupby('社員番号')['hours'].shift(i)

    # 最新行を抽出
    df_worker_lag = df.groupby('社員番号').tail(1).reset_index(drop=True)

    # カラム整形
    lag_cols = [f'hours_{i}_age' for i in range(1, lag + 1)]
    df_worker_lag = df_worker_lag[['社員番号', 'date', 'hours'] + lag_cols]
    df_worker_lag = df_worker_lag.rename(columns={'hours': 'hours_0_age'})

    return df_worker_lag
df_worker_lag = make_worker_hours_lag_features(df_overtime, lag=35)
df_worker_lag.drop('date', axis=1, inplace=True)

In [ ]:
df_udemy_target

In [ ]:
from sentence_transformers import SentenceTransformer

model_name = "hotchpotch/static-embedding-japanese"
model = SentenceTransformer(model_name, device="cpu")

In [ ]:
from pprint import pprint

category_list = ['事業企画・開発・研究', '営業', 'マーケティング', 'コンテンツ・サービス・デザイン', 'プロダクトマネジメント', 'コーポレート管理部門/技術・データ・BPR']
docs = df_udemy["コースタイトル"].unique().tolist()

category = category_list[2]
print(category)
embeddings = model.encode([category] + docs)
similarities = model.similarity(embeddings[0], embeddings[1:])
my_dict = {docs[i]: similarity for i, similarity in enumerate(similarities[0].tolist())}
pprint(dict(sorted(my_dict.items(), key=lambda item: item[1], reverse=True)[:10]), sort_dicts=False)
print()